In [11]:
import joblib
import torch
import numpy as np
from pathlib import Path
from functools import lru_cache
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import sys
import pandas as pd

SOURCE_DIR = Path("/content/drive/MyDrive/Thesis/models")
sys.path.insert(0, str(SOURCE_DIR))

from utils import GOEMOTIONS_LABELS, EKMAN_LABELS
from preprocessing import clean_text_heavy, clean_text_light

MODELS_DIR = Path("/content/drive/MyDrive/Thesis/models")

In [4]:
MODEL_REGISTRY = {
    "phase1":         ("phase1_lr_28class_seed42.joblib", GOEMOTIONS_LABELS, "sklearn"),
    "phase2":         ("phase2_lr_ekman7_heavy_seed42.joblib", EKMAN_LABELS, "sklearn"),
    "phase3":         ("phase3_runs/phase3_distilbert_vanilla_seed42_correct/best_model",  EKMAN_LABELS,      "hf"),
    "phase4_cw":      ("phase4_runs/cross_entropy_runs/phase4_cw_seed42_correct/best_model",                    EKMAN_LABELS,      "hf"),
    "phase4_focal":   ("phase4_runs/focal_loss_runs/phase4_focal_g1_seed42_correct/best_model",             EKMAN_LABELS,      "hf"),
    "phase4_bt":      ("phase4_runs/back_translation_runs/phase4_bt_k3_x3_seed42_correct/best_model",            EKMAN_LABELS,      "hf"),
}


In [5]:
for model_id, (path, labels, kind) in MODEL_REGISTRY.items():
    full_path = MODELS_DIR / path
    print(f"{model_id:<14} | {kind:<7} | exists: {full_path.exists()} | {full_path}")

phase1         | sklearn | exists: True | /content/drive/MyDrive/Thesis/models/phase1_lr_28class_seed42.joblib
phase2         | sklearn | exists: True | /content/drive/MyDrive/Thesis/models/phase2_lr_ekman7_heavy_seed42.joblib
phase3         | hf      | exists: True | /content/drive/MyDrive/Thesis/models/phase3_runs/phase3_distilbert_vanilla_seed42_correct/best_model
phase4_cw      | hf      | exists: True | /content/drive/MyDrive/Thesis/models/phase4_runs/cross_entropy_runs/phase4_cw_seed42_correct/best_model
phase4_focal   | hf      | exists: True | /content/drive/MyDrive/Thesis/models/phase4_runs/focal_loss_runs/phase4_focal_g1_seed42_correct/best_model
phase4_bt      | hf      | exists: True | /content/drive/MyDrive/Thesis/models/phase4_runs/back_translation_runs/phase4_bt_k3_x3_seed42_correct/best_model


In [6]:
@lru_cache(maxsize=8)
def _load(model_id: str):
    path, labels, kind = MODEL_REGISTRY[model_id]
    full_path = MODELS_DIR / path
    if kind == "sklearn":
        return joblib.load(full_path), labels, kind
    else:
        tok = AutoTokenizer.from_pretrained(str(full_path), local_files_only=True)
        mod = AutoModelForSequenceClassification.from_pretrained(str(full_path), local_files_only=True)
        mod.eval()
        if torch.cuda.is_available():
            mod = mod.cuda()
        return (tok, mod), labels, kind


In [7]:
def predict_emotion(text: str, model_id: str = "phase4_focal") -> dict:
    if model_id not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model_id: {model_id}. "
                         f"Available: {list(MODEL_REGISTRY)}")

    artifact, labels, kind = _load(model_id)

    if kind == "sklearn":
        cleaned = clean_text_heavy(text)
        probs = artifact.predict_proba([cleaned])[0]
    else:
        tok, mod = artifact
        cleaned = clean_text_light(text)
        enc = tok(cleaned, return_tensors="pt", truncation=True, max_length=128)

        if torch.cuda.is_available():
            enc = {k: v.cuda() for k, v in enc.items()}

        with torch.no_grad():
            logits = mod(**enc).logits[0].cpu().numpy()

        probs = np.exp(logits - logits.max())
        probs = probs / probs.sum()

    top = int(probs.argmax())

    top_3_indices = np.argsort(probs)[::-1][:3]
    top_3 = [
        {
            "label": labels[i],
            "confidence": float(probs[i])
        }
        for i in top_3_indices
    ]

    return {
        "label": labels[top],
        "confidence": float(probs[top]),
        "top_3": top_3,
    }

In [14]:
import pandas as pd

SENTENCES_TO_TEST = [
    "Thank you so much, this really made my day.",
    "I am NOT happy about this at all.",
]

MODELS_TO_COMPARE = [
    "phase1",
    "phase2",
    "phase3",
    "phase4_cw",
    "phase4_focal",
    "phase4_bt",
]

rows = []

for sentence in SENTENCES_TO_TEST:
    for model_id in MODELS_TO_COMPARE:
        r = predict_emotion(sentence, model_id=model_id)

        rows.append({
            "sentence": sentence,
            "model": model_id,
            "predicted label": r["label"],
            "confidence": f"{r['confidence']:.2f}",
        })

df_all_models = pd.DataFrame(rows)

display(df_all_models)

,sentence,model,predicted label,confidence
0,"Thank you so much, this really made my day.",phase1,gratitude,0.79
1,"Thank you so much, this really made my day.",phase2,joy,0.88
2,"Thank you so much, this really made my day.",phase3,joy,0.99
3,"Thank you so much, this really made my day.",phase4_cw,joy,0.98
4,"Thank you so much, this really made my day.",phase4_focal,joy,0.97
5,"Thank you so much, this really made my day.",phase4_bt,joy,0.97
6,I am NOT happy about this at all.,phase1,joy,0.99
7,I am NOT happy about this at all.,phase2,joy,0.88
8,I am NOT happy about this at all.,phase3,sadness,0.66
9,I am NOT happy about this at all.,phase4_cw,sadness,0.85


In [13]:
import pandas as pd

REHEARSED = [
    "Thank you so much, this really made my day.",          # easy joy
    "I can't believe how thoughtful this is.",              # joy/admiration
    "I'm really sorry for your loss.",                      # sadness / rare-class
    "I am NOT happy about this at all.",                    # negation
    "This is just great. Another delay.",                   # sarcasm
    "Why does this keep happening? I'm losing my mind.",    # anger/fear
    "The funeral is on Friday.",                            # grief
    "I'm so proud of what we built together.",              # pride
    "Wait, what just happened?",                            # surprise/confusion
    "Whatever. Doesn't matter.",                            # neutral / disappointment
]

MODELS_TO_COMPARE = ["phase3", "phase4_focal"]

rows = []

for sentence in REHEARSED:
    for model_id in MODELS_TO_COMPARE:
        r = predict_emotion(sentence, model_id=model_id)

        top_3_text = ", ".join(
            f"{item['label']} ({item['confidence']:.2f})"
            for item in r["top_3"]
        )

        rows.append({
            "sentence": sentence if len(sentence) <= 65 else sentence[:62] + "...",
            "model": model_id,
            "best prediction": r["label"],
            "confidence": f"{r['confidence']:.2f}",
            "top-3 predictions": top_3_text,
        })

df = pd.DataFrame(rows)

display(df)

,sentence,model,best prediction,confidence,top-3 predictions
0,"Thank you so much, this really made my day.",phase3,joy,0.99,"joy (0.99), neutral (0.00), surprise (0.00)"
1,"Thank you so much, this really made my day.",phase4_focal,joy,0.97,"joy (0.97), neutral (0.01), surprise (0.01)"
2,I can't believe how thoughtful this is.,phase3,surprise,0.87,"surprise (0.87), joy (0.05), neutral (0.04)"
3,I can't believe how thoughtful this is.,phase4_focal,surprise,0.84,"surprise (0.84), neutral (0.05), joy (0.04)"
4,I'm really sorry for your loss.,phase3,sadness,0.94,"sadness (0.94), joy (0.02), neutral (0.01)"
5,I'm really sorry for your loss.,phase4_focal,sadness,0.88,"sadness (0.88), joy (0.04), neutral (0.03)"
6,I am NOT happy about this at all.,phase3,sadness,0.66,"sadness (0.66), anger (0.21), neutral (0.05)"
7,I am NOT happy about this at all.,phase4_focal,sadness,0.68,"sadness (0.68), anger (0.20), neutral (0.06)"
8,This is just great. Another delay.,phase3,joy,0.98,"joy (0.98), neutral (0.01), surprise (0.00)"
9,This is just great. Another delay.,phase4_focal,joy,0.92,"joy (0.92), neutral (0.05), surprise (0.01)"


In [12]:
HEAD_TO_HEAD = [
    ("No point in getting rid of Keemun for another Keemun.", "anger"),
    ("Yikes that's cringe", "fear"),
    ("what does dw mean?", "surprise"),
    ("I just need a quiet place to hide.", "neutral"),
    ("This sounds an awful lot like khorne......", "disgust"),
    ("Who needs sciences. It's most fake news they peddle in anyways...", "neutral"),
    ("Thank you so much, this really made my day.", "joy"),
    ("FIRED.", "anger"),
]


def outcome(gold, phase3_label, focal_label):
    if phase3_label == gold and focal_label == gold:
        return "both correct"
    if phase3_label != gold and focal_label == gold:
        return "corrected by focal"
    if phase3_label == gold and focal_label != gold:
        return "new error under focal"
    return "both wrong"


rows = []

for text, gold in HEAD_TO_HEAD:
    phase3_result = predict_emotion(text, model_id="phase3")
    focal_result = predict_emotion(text, model_id="phase4_focal")

    phase3_label = phase3_result["label"]
    focal_label = focal_result["label"]

    phase3_mark = "✓" if phase3_label == gold else "✗"
    focal_mark = "✓" if focal_label == gold else "✗"

    rows.append({
        "sentence": text if len(text) <= 60 else text[:57] + "...",
        "gold": gold,
        "phase3": f"{phase3_label} ({phase3_result['confidence']:.2f}) {phase3_mark}",
        "focal γ=1": f"{focal_label} ({focal_result['confidence']:.2f}) {focal_mark}",
        "outcome": outcome(gold, phase3_label, focal_label),
    })

display(pd.DataFrame(rows).set_index("sentence"))

,gold,phase3,focal γ=1,outcome
sentence,,,,
No point in getting rid of Keemun for another Keemun.,anger,neutral (0.54) ✗,anger (0.52) ✓,corrected by focal
Yikes that's cringe,fear,neutral (0.57) ✗,fear (0.46) ✓,corrected by focal
what does dw mean?,surprise,neutral (0.49) ✗,surprise (0.54) ✓,corrected by focal
I just need a quiet place to hide.,neutral,neutral (0.52) ✓,joy (0.50) ✗,new error under focal
This sounds an awful lot like khorne......,disgust,disgust (0.52) ✓,neutral (0.31) ✗,new error under focal
Who needs sciences. It's most fake news they peddle in an...,neutral,neutral (0.47) ✓,anger (0.51) ✗,new error under focal
"Thank you so much, this really made my day.",joy,joy (0.99) ✓,joy (0.97) ✓,both correct
FIRED.,anger,neutral (0.94) ✗,neutral (0.85) ✗,both wrong
